# 🚀 DataMate (Zalo AI Challenge 2026) — Kaggle Notebook Runner

Notebook này được tối ưu hóa đặc biệt cho môi trường **Kaggle (GPU T4 x 2)** để chạy mô hình mạnh nhất **`Qwen/Qwen2.5-Coder-14B-Instruct` ở chế độ 8-bit (INT8)**.

### ⚙️ Cài đặt bắt buộc trên giao diện Kaggle (Notebook Settings bên phải):
1. **Accelerator**: Chọn **GPU T4 x 2** (2 x 15GB VRAM = 30GB VRAM, hệ thống tự động phân tải đều sang cả 2 card).
2. **Internet**: Chọn **Internet on** (Bắt buộc để nạp thư viện và tải trọng số mô hình từ Hugging Face).
3. **Persistence**: Chọn **Variables and Files**.
4. **Dữ liệu (+ Add Input)**: Vì `competition_data` nằm trong `.gitignore` (không đưa lên GitHub), bạn bấm **+ Add Input** ở góc trên bên phải để thêm Dataset chứa `competition_data` (hoặc `train` và `public_test`). Notebook sẽ **tự động quét và liên kết dữ liệu**.

---
### 💡 Bạn chỉ cần tùy chỉnh tại **Ô Cấu Hình (Cell 1)**:
- **Chọn khoảng câu trên tập Train**: Đặt `TRAIN_RANGE = '1-50'` hoặc `'51-100'` (chạy chính xác từ câu X đến câu Y không random).
- **Chọn số câu trên Public Test**: `PUBLIC_RANGE = '1-100'` hoặc `PUBLIC_BATCH_SIZE = -1` (đủ 1.000 câu).
- **Thứ tự chạy**: Đặt `PRESERVE_ORDER = True` để chạy tuần tự theo đúng thứ tự file `questions.jsonl`.

In [37]:
# ==============================================================================
# 🎯 CẤU HÌNH TẬP TRUNG (CHỌN KHOẢNG CÂU & CHẾ ĐỘ CHẠY TẠI ĐÂY)
# ==============================================================================

# 1. ĐƯỜNG DẪN DỮ LIỆU TRÊN KAGGLE (/kaggle/input/...):
# (Nếu bạn Add Dataset riêng vào Kaggle, hãy điền đường dẫn thư mục chứa train và public_test)
# Mặc định hệ thống cũng sẽ tự động quét phát hiện dữ liệu trong mọi thư mục /kaggle/input/.
KAGGLE_DATA_DIR = "/kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data"

# 2. CẤU HÌNH CHỌN CÂU TRÊN TẬP TRAIN (BƯỚC 5):
# 👉 CÁCH 1 (CHỌN CHÍNH XÁC TỪ CÂU NÀO ĐẾN CÂU NÀO):
#    Điền khoảng "START-END" (ví dụ: "1-50", "51-100", "101-200") -> Chạy đúng khoảng này, KHÔNG random!
# 👉 CÁCH 2 (LẤY MẪU NGẪU NHIÊN): Đặt TRAIN_RANGE = None và điền TRAIN_SAMPLE_SIZE = 30
# 👉 CÁCH 3 (BỎ QUA TẬP TRAIN): Đặt TRAIN_RANGE = None và TRAIN_SAMPLE_SIZE = 0
TRAIN_RANGE = None              # Ví dụ: "1-50", "51-100", "101-200", hoặc None
TRAIN_SAMPLE_SIZE = 50            # Dùng khi TRAIN_RANGE = None (đặt 0 nếu bỏ qua tập train)
TRAIN_MODE = "random"             # "random" (ngẫu nhiên phân tầng) hoặc "sequential" (từ câu 1 đến N)
TRAIN_SEED = 42

# 3. CẤU HÌNH SỐ LƯỢNG CÂU & THỨ TỰ CHẠY TRÊN PUBLIC TEST (BƯỚC 6):
# - PUBLIC_RANGE: Khoảng câu hỏi muốn chạy trên public_test (ví dụ: '1-100', '101-200', hoặc None)
#     • Nếu điền '1-100' -> Chạy chính xác từ câu 1 đến câu 100
#     • Nếu đặt None     -> Chạy theo PUBLIC_BATCH_SIZE bên dưới
PUBLIC_RANGE = None

# - PUBLIC_BATCH_SIZE: Số câu bạn muốn chạy (-1 hoặc 1000 = toàn bộ 1000 câu; hoặc điền 100, 500...)
PUBLIC_BATCH_SIZE = -1

# - PRESERVE_ORDER: Thứ tự chạy các câu hỏi:
#     • True : Chạy TUẦN TỰ từ câu 1 đến câu 1000 đúng theo thứ tự trong questions.jsonl (KHÔNG random, KHÔNG đảo shop)
#     • False: Gom nhóm các câu hỏi cùng shop để chạy liên tiếp (tối ưu tốc độ RAM Cache cao nhất)
PRESERVE_ORDER = True

# 4. CẤU HÌNH MÔ HÌNH LLM:
MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"   # Hoặc 'Qwen/Qwen2.5-Coder-7B-Instruct'
QUANTIZATION = "4bit"                          # "8bit" (mạnh nhất trên GPU T4x2) hoặc "4bit"

# 5. SIÊU THAM SỐ PIPELINE:
TAU_STAR = 0.50                   # Ngưỡng tin cậy hiệu chuẩn tối ưu cho Penalized Accuracy (PA)
TIMEOUT_S = 30                    # Thời gian chờ tối đa khi chạy code trong sandbox (giây)
MAX_NEW_TOKENS = 1536             # Token tối đa cho sinh mã Python

# 6. ĐƯỜNG DẪN MÃ NGUỒN VÀ KẾT QUẢ:
REPO_URL = "https://github.com/Long-uit/datamate.git"
REPO_BRANCH = "phat_commit"       # Nhánh làm việc chính thức ('phat_commit' hoặc 'long-commit')
PROJECT_ROOT = "/kaggle/working/datamate"
RUN_DIR = "/kaggle/working/results/prompt_3"
SUBMISSION_PATH = "/kaggle/working/results/prompt_3/submission.jsonl"

if PUBLIC_RANGE:
    pub_desc = f"từ câu {PUBLIC_RANGE}"
else:
    target_pub = 1000 if (PUBLIC_BATCH_SIZE is None or int(PUBLIC_BATCH_SIZE) <= 0 or int(PUBLIC_BATCH_SIZE) >= 1000) else int(PUBLIC_BATCH_SIZE)
    pub_desc = f"{target_pub} câu"
train_desc = f"từ câu {TRAIN_RANGE}" if TRAIN_RANGE else f"{TRAIN_SAMPLE_SIZE} câu ({TRAIN_MODE})"
print(f"✅ Đã thiết lập: MODEL={MODEL_ID} ({QUANTIZATION}) | PRESERVE_ORDER={PRESERVE_ORDER}")
print(f"🎯 Tập Train: {train_desc} | Public Test: {pub_desc}")

✅ Đã thiết lập: MODEL=Qwen/Qwen2.5-Coder-14B-Instruct (4bit) | PRESERVE_ORDER=True
🎯 Tập Train: 50 câu (random) | Public Test: 1000 câu


In [38]:
# ==============================================================================
# BƯỚC 1: XÁC NHẬN CẤU HÌNH PHẦN CỨNG GPU TRÊN KAGGLE
# ==============================================================================
import os
import torch

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nvidia-smi

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"\n✅ Đã phát hiện {gpu_count} GPU CUDA khả dụng:")
    for i in range(gpu_count):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"   • GPU {i}: {name} ({mem:.1f} GB VRAM)")
    
    if gpu_count >= 2:
        print("🎉 CẤU HÌNH LÝ TƯỞNG: Đang chạy trên GPU T4 x 2! Mô hình 14B 8-bit sẽ tự động chia tải qua cả 2 card.")
    else:
        print("ℹ️ Đang chạy trên 1 GPU. Nếu là P100 (16GB) hoặc T4 đơn, bạn có thể cân nhắc QUANTIZATION='4bit' nếu VRAM chật.")
else:
    print("❌ CẢNH BÁO: Chưa bật GPU! Hãy vào Notebook Settings (bên phải) -> Accelerator -> Chọn 'GPU T4 x 2'.")

Thu Nov 12 18:04:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [39]:
# ==============================================================================
# BƯỚC 2: CLONE MÃ NGUỒN DATAMATE VÀ CÀI ĐẶT THƯ VIỆN BỔ SUNG
# ==============================================================================
import os
import subprocess

if not os.path.exists(PROJECT_ROOT):
    print(f"🔄 Đang clone repository từ {REPO_URL} (ưu tiên nhánh: {REPO_BRANCH})...")
    ret = subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, PROJECT_ROOT])
    if ret.returncode != 0:
        print(f"⚠️ Không tìm thấy nhánh {REPO_BRANCH}, đang thử clone nhánh phat_commit...")
        ret2 = subprocess.run(["git", "clone", "-b", "phat_commit", REPO_URL, PROJECT_ROOT])
        if ret2.returncode != 0:
            print("⚠️ Đang clone nhánh mặc định...")
            !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print(f"🔄 Thư mục {PROJECT_ROOT} đã tồn tại, đang cập nhật mã nguồn mới nhất...")
    %cd {PROJECT_ROOT}
    !git fetch --all
    !git checkout {REPO_BRANCH} || git checkout phat_commit || true
    !git pull origin {REPO_BRANCH} || git pull origin phat_commit || true

%cd {PROJECT_ROOT}
!git status -sb

print("\n📦 Đang cài đặt các thư viện cần thiết...")
!pip install -q -r colab/requirements-colab.txt
!pip install -q -e . --no-deps
print("✅ Hoàn tất thiết lập môi trường DataMate trên Kaggle!")

🔄 Thư mục /kaggle/working/datamate đã tồn tại, đang cập nhật mã nguồn mới nhất...
/kaggle/working/datamate
Already on 'phat_commit'
Your branch is up to date with 'origin/phat_commit'.
From https://github.com/Long-uit/datamate
 * branch            phat_commit -> FETCH_HEAD
Already up to date.
/kaggle/working/datamate
## phat_commit...origin/phat_commit
?? public_test
?? state.db
?? train

📦 Đang cài đặt các thư viện cần thiết...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for datamate (pyproject.toml) ... done
✅ Hoàn tất thiết lập môi trường DataMate trên Kaggle!


In [40]:
# ==============================================================================
# BƯỚC 3: ĐỒNG BỘ DATASET TỪ KAGGLE INPUT VÀO PROJECT
# ==============================================================================
import os
import shutil
from pathlib import Path

# Đường dẫn tuyệt đối
SOURCE_ROOT = Path(
    "/kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data"
)

target_data_dir = Path(PROJECT_ROOT) / "competition_data"
target_data_dir.mkdir(parents=True, exist_ok=True)

print(f"📂 Dataset source: {SOURCE_ROOT}")
print(f"📂 Project target: {target_data_dir}")

# Kiểm tra nguồn dữ liệu
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Dataset tại: {SOURCE_ROOT}"
    )

# Đồng bộ từng split
for split in ["train", "public_test"]:

    src_split = SOURCE_ROOT / split
    dst_split = target_data_dir / split

    if not (src_split / "questions.jsonl").exists():
        print(f"❌ Không tìm thấy dữ liệu {split}: {src_split}")
        continue

    # Xóa liên kết/thư mục đích cũ nếu có
    if dst_split.is_symlink():
        dst_split.unlink()
    elif dst_split.exists():
        if dst_split.is_dir():
            shutil.rmtree(dst_split)
        else:
            dst_split.unlink()

    # Tạo symbolic link đến Dataset gốc
    os.symlink(str(src_split), str(dst_split))

    print(f"✅ Đã liên kết {split}:")
    print(f"   SOURCE: {src_split}")
    print(f"   TARGET: {dst_split}")

# Kiểm tra kết quả
print("\n========== KIỂM TRA DATASET ==========")

for split in ["train", "public_test"]:
    q_file = target_data_dir / split / "questions.jsonl"

    print(f"\n{split.upper()}:")
    print(f"Path: {q_file}")
    print(f"Exists: {q_file.exists()}")

    if q_file.exists():
        with open(q_file, encoding="utf-8") as f:
            count = sum(1 for line in f if line.strip())

        print(f"Questions: {count}")

📂 Dataset source: /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data
📂 Project target: /kaggle/working/datamate/competition_data
✅ Đã liên kết train:
   SOURCE: /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/train
   TARGET: /kaggle/working/datamate/competition_data/train
✅ Đã liên kết public_test:
   SOURCE: /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/public_test
   TARGET: /kaggle/working/datamate/competition_data/public_test

========== KIỂM TRA DATASET ==========

TRAIN:
Path: /kaggle/working/datamate/competition_data/train/questions.jsonl
Exists: True
Questions: 3000

PUBLIC_TEST:
Path: /kaggle/working/datamate/competition_data/public_test/questions.jsonl
Exists: True
Questions: 1000


In [41]:
# ==============================================================================
# BƯỚC 4: KIỂM THỬ NHANH HỆ THỐNG (UNIT TESTS & DRY-RUN KHÔNG DÙNG GPU)
# ==============================================================================
%cd {PROJECT_ROOT}

print("1️⃣ Chạy kiểm thử cấu hình & tính tương thích...")
!python -m pytest tests/test_config.py -q || true

print("\n2️⃣ Chạy thử nghiệm phân tích White-Box 1 câu hỏi (Dry-Run)...)")
if (target_data_dir / "train" / "questions.jsonl").exists():
    dry_cmd = "python tests/inspect_pipeline.py --data_dir competition_data/train --random --dry_run"
    !{dry_cmd}
elif (target_data_dir / "public_test" / "questions.jsonl").exists():
    dry_cmd = "python tests/inspect_pipeline.py --data_dir competition_data/public_test --limit 1 --dry_run"
    !{dry_cmd}
else:
    print("⚠️ Chưa có dữ liệu competition_data. Hãy kiểm tra lại Bước 3!")

print("✅ Khung pipeline chạy thông suốt (Dry-Run hoàn tất)!")

/kaggle/working/datamate
1️⃣ Chạy kiểm thử cấu hình & tính tương thích...
...                                                                      [100%]
3 passed in 0.51s

2️⃣ Chạy thử nghiệm phân tích White-Box 1 câu hỏi (Dry-Run)...)

 🔬  DATAMATE MODULAR WHITE-BOX DIAGNOSTICS & TELEMETRY INSPECTOR
• Question ID   : q_32c6bd8dd5ce
• Business ID   : biz_128
• as_of_date    : 2026-09-07
• Câu hỏi Gốc   : "Tháng 4/2025 bị huỷ mấy đơn?"
• Đáp án Chuẩn  : 31 (Type: number)

┌──────────────────────────────────────────────────────────────────────────────┐
│ [MỤC 1] XỬ LÝ TIẾNG VIỆT TỪ VIẾT TẮT SANG DẠNG CHUẨN (TRƯỚC & SAU)          │
├──────────────────────────────────────────────────────────────────────────────┤
│ • TRƯỚC (Gốc)      : "Tháng 4/2025 bị huỷ mấy đơn?"
│ • SAU (Chuẩn hóa)  : "Tháng 4/2025 bị huỷ mấy đơn?"
│ • Viết tắt đã mở   : (Không chứa từ viết tắt trong từ điển)
└──────────────────────────────────────────────────────────────────────────────┘

--- [BƯỚC 1] NẠP DỮ LIỆU CỬA 

In [42]:
from pathlib import Path
import os

project = Path("/kaggle/working/datamate")
source = Path("/kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data")

print("========== KIỂM TRA DATASET ==========")

for split in ["train", "public_test"]:
    src = source / split
    dst = project / "competition_data" / split
    q_src = src / "questions.jsonl"
    q_dst = dst / "questions.jsonl"

    print(f"\n[{split.upper()}]")
    print("Source exists:", q_src.exists())
    print("Target exists:", q_dst.exists())
    print("Target directory exists:", dst.exists())
    print("Target is symlink:", dst.is_symlink())

    if dst.is_symlink():
        print("Symlink points to:", os.readlink(dst))

    if q_src.exists():
        print("Source file size:", q_src.stat().st_size)

    if q_dst.exists():
        print("Target file size:", q_dst.stat().st_size)

print("\nCurrent directory:", Path.cwd())

========== KIỂM TRA DATASET ==========

[TRAIN]
Source exists: True
Target exists: True
Target directory exists: True
Target is symlink: True
Symlink points to: /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/train
Source file size: 504081
Target file size: 504081

[PUBLIC_TEST]
Source exists: True
Target exists: True
Target directory exists: True
Target is symlink: True
Symlink points to: /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/public_test
Source file size: 163342
Target file size: 163342

Current directory: /kaggle/working/datamate


In [ ]:
# ==============================================================================
# BƯỚC 5: ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH 14B TRÊN TẬP TRAIN (THEO KHOẢNG CÂU HOẶC MẪU)
# ==============================================================================
import os
import gc
import torch
from pathlib import Path
from IPython.display import Image, display

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

%cd {PROJECT_ROOT}
train_eval_dir = Path(RUN_DIR) / "train_eval"
train_eval_dir.mkdir(parents=True, exist_ok=True)

train_pred_file = train_eval_dir / "train_predictions.jsonl"
train_log_dir = train_eval_dir / "logs"
train_rep_file = train_eval_dir / "evaluation_report.json"
train_md_file = train_eval_dir / "evaluation_report.md"
train_chart_file = train_eval_dir / "evaluation_dashboard.png"

if not (Path("competition_data/train/questions.jsonl").exists()):
    print("⏭️ Không tìm thấy competition_data/train/questions.jsonl. Tự động bỏ qua Bước 5.")
elif not TRAIN_RANGE and (TRAIN_SAMPLE_SIZE is not None and int(TRAIN_SAMPLE_SIZE) == 0):
    print("⏭️ Đã cấu hình bỏ qua kiểm định trên tập TRAIN (TRAIN_SAMPLE_SIZE = 0).")
else:
    flags = []
    if TRAIN_RANGE:
        flags.append(f"--range {TRAIN_RANGE}")
        print(f"🎯 Đang đánh giá CHÍNH XÁC từ câu {TRAIN_RANGE} của tập TRAIN...")
    elif TRAIN_MODE == "random" and int(TRAIN_SAMPLE_SIZE) > 0:
        flags.append(f"--sample {TRAIN_SAMPLE_SIZE} --stratified")
        print(f"🎯 Đang lấy mẫu ngẫu nhiên phân tầng {TRAIN_SAMPLE_SIZE} câu...")
    elif int(TRAIN_SAMPLE_SIZE) > 0:
        flags.append(f"--limit {TRAIN_SAMPLE_SIZE}")
        print(f"🎯 Đang chạy tuần tự {TRAIN_SAMPLE_SIZE} câu đầu tiên...")
    
    if PRESERVE_ORDER:
        flags.append("--preserve_order")
    extra_flags = " ".join(flags)

    print(f"🚀 Bắt đầu đánh giá mô hình {MODEL_ID} ({QUANTIZATION}) trên tập TRAIN...")
    train_cmd = (
        f"python tests/inspect_pipeline.py "
        f"--batch "
        f"--with_llm "
        f"--data_dir competition_data/train "
        f"--model_id '{MODEL_ID}' "
        f"--quantization '{QUANTIZATION}' "
        f"--seed {TRAIN_SEED} "
        f"--tau_star {TAU_STAR} "
        f"--timeout {TIMEOUT_S} "
        f"--max_new_tokens {MAX_NEW_TOKENS} "
        f"--no_fill_abstain "
        f"--output_file '{train_pred_file}' "
        f"--log_dir '{train_log_dir}' "
        f"--report_file '{train_rep_file}' "
        f"--md_file '{train_md_file}' "
        f"--chart_file '{train_chart_file}' "
        f"{extra_flags}"
    )
    !{train_cmd}

if train_chart_file.exists():
    print("\n📊 Biểu đồ trực quan kết quả đánh giá (Evaluation Dashboard):")
    display(Image(filename=str(train_chart_file)))

/kaggle/working/datamate
🎯 Đang lấy mẫu ngẫu nhiên phân tầng 50 câu...
🚀 Bắt đầu đánh giá mô hình Qwen/Qwen2.5-Coder-14B-Instruct (4bit) trên tập TRAIN...
DATAMATE — PIPELINE SUY LUẬN, SO KHỚP ĐÁP ÁN & TRỰC QUAN HÓA TOÀN TRÌNH
• Dữ liệu đầu vào       : /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/train/questions.jsonl (3000 câu)
• Số câu thực hiện chạy : 50 câu (Limit: None)
• File đáp án chuẩn     : /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/train/answers.jsonl
• File kết quả JSONL    : /kaggle/working/results/prompt_3/train_eval/train_predictions.jsonl
• Ảnh Dashboard thống kê: /kaggle/working/results/prompt_3/train_eval/evaluation_dashboard.png
[Khởi tạo] Đang nạp mô hình Qwen/Qwen2.5-Coder-14B-Instruct (backend=transformers)...
[LLMClient] Bắt đầu nạp mô hình: Qwen/Qwen2.5-Coder-14B-Instruct (backend=transformers)...
[LLMClient] 🖥️ Phát hiện phần cứng: 2 GPU (Tesla T4, Tesla T4) | Tổng VRAM: 29.1 GB
[LLMClient] Áp dụng lượng tử 

In [30]:
import shutil
from pathlib import Path

# Thư mục cần nén
SOURCE_DIR = Path("/kaggle/working/results/prompt_2")

# Đường dẫn file ZIP đầu ra
ZIP_PATH = Path("/kaggle/working/results/prompt_2.zip")

# Tiến hành nén
shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=SOURCE_DIR.parent,
    base_dir=SOURCE_DIR.name
)

print(f"✅ Đã nén thành công: {ZIP_PATH}")
print(f"📦 Kích thước: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

✅ Đã nén thành công: /kaggle/working/results/prompt_2.zip
📦 Kích thước: 0.08 MB


In [16]:
from pathlib import Path

print("========== KAGGLE INPUT ==========")

input_dir = Path("/kaggle/input")

for path in input_dir.iterdir():
    print(f"📁 {path}")

print("\n========== TÌM QUESTIONS.JSONL ==========")

for file in input_dir.rglob("questions.jsonl"):
    print(f"✅ Found: {file}")

print("\n========== PROJECT DATA ==========")

project_data = Path("/kaggle/working/datamate/competition_data")

print(f"Exists: {project_data.exists()}")

if project_data.exists():
    for path in project_data.iterdir():
        print(path)

========== KAGGLE INPUT ==========
📁 /kaggle/input/notebooks
📁 /kaggle/input/datasets

========== TÌM QUESTIONS.JSONL ==========
✅ Found: /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/public_test/questions.jsonl
✅ Found: /kaggle/input/datasets/minhphatgdgoc/competition-data/competition_data/train/questions.jsonl

========== PROJECT DATA ==========
Exists: True
/kaggle/working/datamate/competition_data/train
/kaggle/working/datamate/competition_data/public_test


In [27]:
from IPython.display import display, Markdown

report_path = "/kaggle/working/results/train_eval/evaluation_report.md"
with open(report_path, "r", encoding="utf-8") as f:
    content = f.read()

# Hiển thị báo cáo chi tiết
display(Markdown(content))

# 📊 Báo cáo Đánh giá Toàn diện Hệ thống DataMate (Batch Evaluation Report)

- **Mô hình sử dụng (`model_id`)**: `Qwen/Qwen2.5-Coder-14B-Instruct`
- **Tổng số câu hỏi đánh giá**: `50` câu
- **Tổng thời gian thực thi**: `3863.54s` (Trung bình `77.27s/câu`)

## 1. Tổng hợp Điểm Penalized Accuracy (PA)

| Chỉ số Đánh giá | Giá trị | Tỷ lệ / Ghi chú |
| :--- | :---: | :--- |
| **Penalized Accuracy (PA)** | **`-0.1400`** | Công thức: $(Correct - Wrong) / N$ |
| **Số câu Đúng (+1 điểm)** | `7` | `14.0%` |
| **Số câu Sai (-1 điểm)** | `14` | `28.0%` |
| **Số câu Từ chối / Abstain (0 điểm)** | `29` | `58.0%` |
| **Accuracy (khi trả lời)** | `33.3%` | $Correct / (Correct + Wrong)$ |
| **Coverage (Độ bao phủ trả lời)** | `42.0%` | $(Correct + Wrong) / N$ |

## 2. Phân rã Kết quả theo Kiểu Đáp án (Answer Type Breakdown)

| Kiểu đáp án (`type`) | Tổng số câu | Đúng (+1) | Sai (-1) | Abstain (0) | Điểm PA | Độ trễ TB (ms) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| `date` | 3 | 0 | 1 | 2 | **`-0.333`** | 53608.3 |
| `list` | 4 | 0 | 1 | 3 | **`-0.250`** | 63982.5 |
| `number` | 31 | 3 | 7 | 21 | **`-0.129`** | 84408.4 |
| `percent` | 8 | 1 | 5 | 2 | **`-0.500`** | 179985.1 |
| `string` | 1 | 0 | 0 | 1 | **`+0.000`** | 52129.0 |
| `unanswerable` | 3 | 3 | 0 | 0 | **`+1.000`** | 291.0 |

## 3. Thống kê Nguyên nhân Lỗi & Lý do Từ chối (Error & Abstention Taxonomy)

| Nguyên nhân / Chẩn đoán | Số lượng câu |
| :--- | :---: |
| Sai lệch logic/số học: Chưa khớp đáp án chuẩn | `11` |
| Runtime/Sanity Error: Traceback (most recent call last): | `2` |
| Sai lệch logic/số học: CÓ THỂ QUÊN LỌC ĐƠN HỦY / TRẢ LẠI (Giá trị dự đoán cao hơn chuẩn 5-30%). | `2` |
| Runtime/Sanity Error: Silent Zero: Kết quả tính toán bằng 0 bất thường cho câu hỏi doanh thu / số lượng / trung  | `1` |
| Sai lệch logic/số học: CÓ THỂ LỌC THIẾU DÒNG HOẶC ÁP DỤNG TRÙNG ĐIỀU KIỆN (Dự đoán thấp hơn chuẩn 5-30%). | `1` |

---

## 4. Báo cáo Phân tích Chi tiết Từng Câu hỏi (Per-Question Analysis)

### 🛡️ Câu 1: `q_026fabde15fc` (`biz_103`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Cho hỏi tổng doanh thu quý 2/2026 là bao nhiêu?
- **Ngữ cảnh**: `as_of_date = 2026-07-09` | `intent = aggregation` | Khoảng ngày: `2026-04-01 -> 2026-06-30` (Quý 2/2026)
- **Thực thể khớp (Entity Linking)**: `2 bao`, `1 bao`, `2 thùng`, `2 bó`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `518445500` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`166.3ms` | Grounding=`71.4ms` | LLM=`28388.2ms` | Thực thi=`11734.0ms` | Normalizer=`0.2ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`0%` (Phạt `-0.35`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ❌ Câu 2: `q_02a28b3629be` (`biz_103`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Cho hỏi số đơn bị huỷ trong tháng 8 năm 2025 là bao nhiêu?
- **Ngữ cảnh**: `as_of_date = 2026-07-09` | `intent = cancelled_orders` | Khoảng ngày: `2025-08-01 -> 2025-08-31` (Tháng 8/2025)
- **Thực thể khớp (Entity Linking)**: `2 bao`, `1 bao`, `8 lon`, `8 bó`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date', 'Trạng thái', 'Mã đơn hàng']
- **Dự đoán (Pred)**: `0` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `32` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.85` | **Số vòng tự sửa lỗi (Self-Correction)**: `1`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`145.0ms` | Grounding=`72.5ms` | LLM=`59113.4ms` | Thực thi=`39241.2ms` | Normalizer=`3.1ms` | Retry=`1`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `32.0` (100.0%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Câu hỏi đếm đơn bị hủy nhưng code không lọc điều kiện trạng thái hủy` -> Kết quả thử lại: `Thành công`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df = get_sales_table(data)
    date_col = get_date_col(df)
    df['clean_date'] = clean_date(df[date_col])
    
    # Lọc dữ liệu trong tháng 8 năm 2025
    filtered_df = df[(df['clean_date'] >= pd.Timestamp('2025-08-01')) & (df['clean_date'] <= pd.Timestamp('2025-08-31 23:59:59'))]
    
    # Lọc đơn bị hủy
    cancelled_orders = filtered_df[filtered_df['Trạng thái'] == 'đã hủy']
    
    # Đếm số đơn bị hủy
    result = int(cancelled_orders['Mã đơn hàng'].dropna().nunique())
    
    return result
```
</details>

### 🛡️ Câu 3: `q_0483a8398dfe` (`biz_103`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: dthu 3 tháng liền nhau kết thúc ở tháng 12/2025 là bao nhiêu mỗi tháng, từ tháng cũ đến tháng mới?
- **Câu hỏi mở rộng viết tắt**: doanh thu 3 tháng liền nhau kết thúc ở tháng 12/2025 là bao nhiêu mỗi tháng, từ tháng cũ đến tháng mới?
- **Ngữ cảnh**: `as_of_date = 2026-07-09` | `intent = aggregation` | Khoảng ngày: `2025-12-01 -> 2025-12-31` (Tháng 12/2025)
- **Thực thể khớp (Entity Linking)**: `3 thùng`, `2 bao`, `1 bao`, `3 chai`, `12,000`, `12,500`, `3 bó`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `[173866200, 156289700, 158205600]` (Kiểu: `list`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`161.2ms` | Grounding=`122.3ms` | LLM=`29891.5ms` | Thực thi=`11688.7ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 4: `q_042171000b2c` (`biz_104`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Tháng 2/2026 có bao nhiêu đơn hàng bị huỷ?
- **Ngữ cảnh**: `as_of_date = 2026-06-02` | `intent = cancelled_orders` | Khoảng ngày: `2026-02-01 -> 2026-02-28` (Tháng 2/2026)
- **Thực thể khớp (Entity Linking)**: `2.0`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `41` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`144.5ms` | Grounding=`19.1ms` | LLM=`30482.1ms` | Thực thi=`16549.5ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 5: `q_02ed16421456` (`biz_105`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: tháng 8/2025 quán bán đc bn ly size m
- **Câu hỏi mở rộng viết tắt**: tháng 8/2025 quán bán được bao nhiêu ly size m
- **Ngữ cảnh**: `as_of_date = 2026-06-24` | `intent = quantity_sum` | Khoảng ngày: `2025-08-01 -> 2025-08-31` (Tháng 8/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date']
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `5604` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `2`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`674.5ms` | Grounding=`11.8ms` | LLM=`157151.9ms` | Thực thi=`180418.5ms` | Normalizer=`0.0ms` | Retry=`2`
- **⚠️ Nguyên nhân Lỗi / Lý do chặn**: `Silent Zero: Kết quả tính toán bằng 0 bất thường cho câu hỏi doanh thu / số lượng / trung bình`
- **🛡️ Lý do Cổng Abstention**: `Sanity Error: Silent Zero: Kết quả tính toán bằng 0 bất thường cho câu hỏi doanh thu / số lượng / trung bình`
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Silent Zero: Kết quả tính toán bằng 0 bất thường cho câu hỏi doanh thu / số lượng / trung bình` -> Kết quả thử lại: `Silent Zero: Kết quả tính toán bằng 0 bất thường cho câu hỏi doanh thu / số lượng / trung bình`
  - Vòng 2: Lỗi đầu vào `Silent Zero: Kết quả tính toán bằng 0 bất thường cho câu hỏi doanh thu / số lượng / trung bình` -> Kết quả thử lại: `Silent Zero: Kết quả tính toán bằng 0 bất thường cho câu hỏi doanh thu / số lượng / trung bình`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df_sales = get_sales_table(data)
    if df_sales is None or df_sales.empty:
        return 0

    date_col = get_date_col(df_sales)
    qty_col = get_qty_col(df_sales)
    item_col = get_item_col(df_sales)
    if not date_col or not qty_col or not item_col:
        return 0

    df_sales['clean_date'] = clean_date(df_sales[date_col])
    cols = df_sales.columns.tolist()
    status_col = next((c for c in cols if any(k in str(c).lower() for k in ['trạng thái', 'status', 'ghi chú tt', 'tình trạng'])), None)
    if status_col:
        df_sales = df_sales[~df_sales[status_col].astype(str).str.lower().str.contains(r'hủy|huỷ|huy|trả|tra|cancel|return')]

    mask_item = df_sales[item_col].astype(str).str.contains(r'size m', case=False, regex=True, na=False)
    mask_date = (df_sales['clean_date'] >= pd.Timestamp('2025-08-01')) & (df_sales['clean_date'] <= pd.Timestamp('2025-08-31 23:59:59'))
    df_filtered = df_sales[mask_item & mask_date]
    if df_filtered.empty:
        return 0

    return int(round(clean_num(df_filtered[qty_col]).sum()))
```
</details>

### 🛡️ Câu 6: `q_03cf8b6c4b16` (`biz_105`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: thang 3/2026 hang nuoc ep dua hau (l) thu ve bao nhieu tien ha
- **Ngữ cảnh**: `as_of_date = 2026-06-24` | `intent = aggregation` | Khoảng ngày: `2026-03-01 -> 2026-03-31` (Tháng 3/2026)
- **Thực thể khớp (Entity Linking)**: `Nước ép dưa hấu (L)`, `Nước ép dưa hấu (M)`, `Nước ép cam (L)`, `Nước ép cam (M)`, `Latte (L)`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `4012000` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`617.7ms` | Grounding=`43.6ms` | LLM=`42286.6ms` | Thực thi=`46314.7ms` | Normalizer=`0.2ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`0%` (Phạt `-0.35`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 7: `q_0266c6b24cbf` (`biz_106`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Tháng 5/2026 giá nhập bình quân một hộp Bánh croissant là bao nhiêu vậy?
- **Ngữ cảnh**: `as_of_date = 2026-09-14` | `intent = purchase_import` | Khoảng ngày: `2026-05-01 -> 2026-05-31` (Tháng 5/2026)
- **Thực thể khớp (Entity Linking)**: `BÁNH CROISSANT`, `bánh`, `Bánh croissant`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `92897` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`1239.0ms` | Grounding=`144.6ms` | LLM=`92126.4ms` | Thực thi=`70013.0ms` | Normalizer=`7.6ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`0%` (Phạt `-0.35`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 8: `q_047c8277cc35` (`biz_108`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Giá trị trung bình một đơn tháng 10/2025 là bao nhiêu nhỉ?
- **Ngữ cảnh**: `as_of_date = 2026-05-29` | `intent = average_order_value` | Khoảng ngày: `2025-10-01 -> 2025-10-31` (Tháng 10/2025)
- **Thực thể khớp (Entity Linking)**: `gia vị`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `158110` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`31317.7ms` | Grounding=`68.3ms` | LLM=`40822.8ms` | Thực thi=`31117.8ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ❌ Câu 9: `q_058e9b27fb45` (`biz_108`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: gia ban mot goi mi hao hao tom chua cay vao ngay 21/04/2026 la bnhiu ta
- **Câu hỏi mở rộng viết tắt**: gia ban mot goi mi hao hao tom chua cay vao ngay 21/04/2026 la bao nhiêu ta
- **Ngữ cảnh**: `as_of_date = 2026-05-29` | `intent = price_lookup` | Khoảng ngày: `2026-04-21 -> 2026-04-21` (Ngày 21/04/2026 (2026-04-21))
- **Thực thể khớp (Entity Linking)**: `Mì Hảo Hảo tôm chua cay`, `mì gói`, `gia vị`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date']
- **Dự đoán (Pred)**: `0` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `4500` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.78` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`192.8ms` | Grounding=`127.9ms` | LLM=`39034.2ms` | Thực thi=`30414.2ms` | Normalizer=`2.7ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`50%` (Phạt `-0.17`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `4500.0` (100.0%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
import pandas as pd

def solution(data):
    target_date = pd.Timestamp('2026-04-21')
    product_name = 'Mì Hảo Hảo tôm chua cay'
    
    # Get sales table
    df = get_sales_table(data)
    
    # Find relevant columns
    d_c = get_date_col(df)
    a_c = get_amount_col(df)
    i_c = get_item_col(df)
    
    if d_c and a_c and i_c:
        df['clean_date'] = clean_date(df[d_c])
        matched = df[df['clean_date'].dt.date == target_date]
        matched_product = matched[matched[i_c].astype(str).str.contains(product_name, case=False, na=False)]
        
        if not matched_product.empty:
            return int(round(clean_num(matched_product[a_c]).sum()))
    
    return 0
```
</details>

### 🛡️ Câu 10: `q_05bf323fdee2` (`biz_108`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: hom qua cua hang ban duoc bao nhiu tien
- **Ngữ cảnh**: `as_of_date = 2026-05-11` | `intent = aggregation` | Khoảng ngày: `2026-05-10 -> 2026-05-10` (Hôm qua (2026-05-10))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `6545000` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`189.6ms` | Grounding=`49.8ms` | LLM=`33340.1ms` | Thực thi=`30834.3ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ✅ Câu 11: `q_0504f61cae69` (`biz_110`) — Điểm PA: `+1.0` (HOÀN TOÀN TRÙNG KHỚP (CORRECT - +1.0))
- **Câu hỏi gốc**: e bua qua cua hang ban duoc may don
- **Ngữ cảnh**: `as_of_date = 2025-10-03` | `intent = aggregation` | Khoảng ngày: `2025-10-02 -> 2025-10-02` (Hôm qua (2025-10-02))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date']
- **Dự đoán (Pred)**: `93` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `93` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`34.4ms` | Grounding=`1.8ms` | LLM=`61111.4ms` | Thực thi=`2187.3ms` | Normalizer=`0.6ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    from datetime import datetime
    import pandas as pd
    
    START_DATE = '2025-10-02'
    END_DATE = '2025-10-02'
    
    df_sales = get_sales_table(data)
    if df_sales is None or df_sales.empty:
        return 0

    date_col = get_date_col(df_sales)
    if not date_col:
        return 0

    df_sales['clean_date'] = clean_date(df_sales[date_col])
    mask = (df_sales['clean_date'] >= pd.Timestamp(START_DATE)) & (df_sales['clean_date'] <= pd.Timestamp(END_DATE))
    df_sales = df_sales[mask]

    order_col = next((c for c in df_sales.columns if any(k in str(c).lower() for k in ['mã đơn', 'ma don', 'mã hoá đơn', 'mã hd', 'order_id', 'order id', 'mã chứng từ', 'mã gd', 'mã'])), None)
    if not order_col:
        return 0

    valid_orders = df_sales[order_col].dropna().nunique()
    return int(valid_orders)
```
</details>

### ❌ Câu 12: `q_0279f675859e` (`biz_111`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: thang 7/2025, cac khach thuoc nhom than thiet mua tong cong bao nhiu tien hang
- **Ngữ cảnh**: `as_of_date = 2026-08-19` | `intent = aggregation` | Khoảng ngày: `2025-07-01 -> 2025-07-31` (Tháng 7/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date']
- **Dự đoán (Pred)**: `251060500` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `46871000` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`571.5ms` | Grounding=`9.2ms` | LLM=`20806.0ms` | Thực thi=`11472.1ms` | Normalizer=`0.7ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `204189500.0` (435.6414414029997%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df_sales = get_sales_table(data)
    if df_sales is None or df_sales.empty:
        return 0

    date_col = get_date_col(df_sales)
    amt_col = get_amount_col(df_sales)
    item_col = get_item_col(df_sales)
    if not date_col or not amt_col:
        return 0

    df_sales['clean_date'] = clean_date(df_sales[date_col])
    cols = df_sales.columns.tolist()
    status_col = next((c for c in cols if any(k in str(c).lower() for k in ['trạng thái', 'status', 'ghi chú tt', 'tình trạng', 'xử lý'])), None)
    if status_col:
        df_sales = df_sales[~df_sales[status_col].astype(str).str.lower().str.contains(r'hủy|huỷ|huy|trả|cancel', regex=True, na=False)]

    mask = (df_sales['clean_date'].dt.year == 2026) & (df_sales['clean_date'].dt.month == 3)
    df_sales = df_sales[mask]

    if item_col:
        mask_prod = df_sales[item_col].astype(str).str.contains('nước ép dưa hấu', case=False, na=False) & df_sales[item_col].astype(str).str.contains(r'\(l\)|\bl\b', case=False, regex=True, na=False)
        df_sales = df_sales[mask_prod]

    if df_sales.empty:
        return 0
    return int(round(clean_num(df_sales[amt_col]).sum()))
```
</details>

### ✅ Câu 13: `q_0271d9f39fde` (`biz_112`) — Điểm PA: `+1.0` (HOÀN TOÀN TRÙNG KHỚP (CORRECT - +1.0))
- **Câu hỏi gốc**: tháng 5/2026 cửa hàng có nhiêu % đơn bị huỷ
- **Ngữ cảnh**: `as_of_date = 2026-07-31` | `intent = rate_percentage` | Khoảng ngày: `2026-05-01 -> 2026-05-31` (Tháng 5/2026)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date', 'Mã đơn hàng']
- **Dự đoán (Pred)**: `2.19` (Kiểu: `percent`) | **Đáp án Chuẩn (Gold)**: `2.19` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.80` | **Số vòng tự sửa lỗi (Self-Correction)**: `1`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`386.3ms` | Grounding=`17.9ms` | LLM=`84860.6ms` | Thực thi=`78691.0ms` | Normalizer=`0.8ms` | Retry=`1`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Câu hỏi kiểu percent nhưng code không thực hiện phép chia tính tỷ lệ; Câu hỏi đếm đơn bị hủy nhưng code không lọc điều kiện trạng thái hủy` -> Kết quả thử lại: `Thành công`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df = get_sales_table(data)
    date_col = get_date_col(df)
    status_col = next((col for col in df.columns if 'Trạng thái' in col), None)
    
    if not date_col or not status_col:
        return 0.0
    
    df['clean_date'] = clean_date(df[date_col])
    filtered_df = df[(df['clean_date'] >= pd.Timestamp('2026-05-01')) & (df['clean_date'] <= pd.Timestamp('2026-05-31 23:59:59'))]
    
    total_orders = int(filtered_df['Mã đơn hàng'].dropna().nunique())
    cancelled_orders = int(filtered_df[filtered_df[status_col].str.contains('hủy|cancel', case=False, na=False)]['Mã đơn hàng'].dropna().nunique())
    
    if total_orders == 0:
        return 0.0
    
    cancellation_rate = (cancelled_orders / total_orders) * 100
    return float(round(cancellation_rate, 2))
```
</details>

### ❌ Câu 14: `q_02a70ea566f1` (`biz_112`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Top 3 sp bán nhiều nhất tháng 3 năm 2026 là những gì?
- **Câu hỏi mở rộng viết tắt**: Top 3 sản phẩm bán nhiều nhất tháng 3 năm 2026 là những gì?
- **Ngữ cảnh**: `as_of_date = 2026-07-31` | `intent = ranking_topk` | Khoảng ngày: `2026-03-01 -> 2026-03-31` (Tháng 3/2026)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date', 'clean_amount']
- **Dự đoán (Pred)**: `['Trà sữa trân châu (M)', 'Trà đào cam sả (M)', 'Bạc xỉu (M)']` (Kiểu: `list`) | **Đáp án Chuẩn (Gold)**: `['Cà phê sữa (M)', 'Trà đào cam sả (M)', 'Cà phê đen (M)']` (Kiểu: `list`)
- **Độ tin cậy ($C$)**: `0.90` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`327.6ms` | Grounding=`20.6ms` | LLM=`78616.6ms` | Thực thi=`32721.8ms` | Normalizer=`0.6ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
import pandas as pd

def solution(data):
    # Lấy bảng bán hàng
    df_sales = get_sales_table(data)
    
    # Tìm cột ngày và tiền
    d_c = get_date_col(df_sales)
    a_c = get_amount_col(df_sales)
    
    # Tạo cột clean_date
    df_sales['clean_date'] = clean_date(df_sales[d_c])
    
    # Lọc dữ liệu trong tháng 3 năm 2026
    mask = (df_sales['clean_date'] >= pd.Timestamp('2026-03-01')) & (df_sales['clean_date'] <= pd.Timestamp('2026-03-31 23:59:59'))
    df_period = df_sales[mask]
    
    # Tìm cột sản phẩm
    prod_col = get_item_col(df_sales)
    
    # Tính tổng số lượng bán
    df_period['clean_amount'] = clean_num(df_period[a_c])
    grp = df_period.groupby(prod_col, as_index=False)['clean_amount'].sum()
    
    # Sắp xếp theo số lượng bán giảm dần
    grp = grp.sort_values(by='clean_amount', ascending=False)
    
    # Trả về top 3 sản phẩm bán nhiều nhất
    return grp.head(3)[prod_col].astype(str).tolist()
```
</details>

### 🛡️ Câu 15: `q_04f73179d0fe` (`biz_112`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: thang 10/2025 cac mat hang nhom tra sua thu duoc total bn tien
- **Câu hỏi mở rộng viết tắt**: thang 10/2025 cac mat hang nhom tra sua thu duoc total bao nhiêu tien
- **Ngữ cảnh**: `as_of_date = 2026-07-31` | `intent = aggregation` | Khoảng ngày: `2025-10-01 -> 2025-10-31` (Tháng 10/2025)
- **Thực thể khớp (Entity Linking)**: `trà`, `trà sữa`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `63445000` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`355.5ms` | Grounding=`19.2ms` | LLM=`34128.9ms` | Thực thi=`32255.7ms` | Normalizer=`0.3ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`0%` (Phạt `-0.35`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 16: `q_052517924e03` (`biz_112`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Tháng 6/2025 có mấy ngày doanh thu đạt 9.800.000 đồng trở lên?
- **Ngữ cảnh**: `as_of_date = 2026-07-31` | `intent = aggregation` | Khoảng ngày: `2025-06-01 -> 2025-06-30` (Tháng 6/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `14` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`339.3ms` | Grounding=`15.5ms` | LLM=`34760.4ms` | Thực thi=`32439.5ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ✅ Câu 17: `q_04f72e9f3a9e` (`biz_113`) — Điểm PA: `+1.0` (HOÀN TOÀN TRÙNG KHỚP (CORRECT - +1.0))
- **Câu hỏi gốc**: tong revenue trong ngay hom nay la bao nhiu
- **Ngữ cảnh**: `as_of_date = 2025-12-29` | `intent = aggregation` | Khoảng ngày: `2025-12-29 -> 2025-12-29` (Hôm nay (2025-12-29))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `30226400` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `30226400` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`6158.8ms` | Grounding=`21.3ms` | LLM=`22518.5ms` | Thực thi=`7403.5ms` | Normalizer=`0.3ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    return revenue(data, start_date='2025-12-29', end_date='2025-12-29', as_of_date='2025-12-29')
```
</details>

### 🛡️ Câu 18: `q_02f8f6235ddb` (`biz_115`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: coi gium trong thang 5 nam 2025 cua hang ban duoc may kg thep cuon phi 6 hoa phat
- **Ngữ cảnh**: `as_of_date = 2026-05-16` | `intent = quantity_sum` | Khoảng ngày: `2025-05-01 -> 2025-05-31` (Tháng 5/2025)
- **Thực thể khớp (Entity Linking)**: `Thép cuộn phi 6 Hòa Phát`, `Thép cuộn phi 8 Hòa Phát`, `sắt thép`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `130` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`201.2ms` | Grounding=`112.3ms` | LLM=`36632.4ms` | Thực thi=`15754.3ms` | Normalizer=`0.2ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`0%` (Phạt `-0.35`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 19: `q_040ffbb6869a` (`biz_115`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: cho hoi dem so don da huy trong thang 2/2026
- **Ngữ cảnh**: `as_of_date = 2026-05-16` | `intent = cancelled_orders` | Khoảng ngày: `2026-02-01 -> 2026-02-28` (Tháng 2/2026)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `14` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`168.2ms` | Grounding=`29.5ms` | LLM=`34904.9ms` | Thực thi=`15240.5ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 20: `q_0465a6852f62` (`biz_115`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: gias trị trung bình một đơn thangs 4/2026 là bao nhiêu
- **Câu hỏi mở rộng viết tắt**: giá trị trung bình một đơn tháng 4/2026 là bao nhiêu
- **Ngữ cảnh**: `as_of_date = 2026-05-16` | `intent = average_order_value` | Khoảng ngày: `2026-04-01 -> 2026-04-30` (Tháng 4/2026)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `1368446` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`166.7ms` | Grounding=`53.5ms` | LLM=`39282.7ms` | Thực thi=`15842.7ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 21: `q_056e80ba5099` (`biz_115`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Hôm nay bán được mấy đơn hàng?
- **Ngữ cảnh**: `as_of_date = 2025-08-12` | `intent = count_orders` | Khoảng ngày: `2025-08-12 -> 2025-08-12` (Hôm nay (2025-08-12))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `28` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`166.5ms` | Grounding=`31.7ms` | LLM=`38018.4ms` | Thực thi=`15760.8ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 22: `q_05aca83bd353` (`biz_115`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Trong giai đoạn từ 20/10/2025 đến hết 26/10/2025, bao nhiêu mặt hàng bị ế nhỉ?
- **Ngữ cảnh**: `as_of_date = 2026-05-16` | `intent = aggregation` | Khoảng ngày: `None -> 2026-05-16` (Mốc thời gian tham chiếu (hôm nay): 2026-05-16)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `4` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`171.4ms` | Grounding=`70.9ms` | LLM=`35224.1ms` | Thực thi=`14962.6ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ✅ Câu 23: `q_03d662023a15` (`biz_116`) — Điểm PA: `+1.0` (HOÀN TOÀN TRÙNG KHỚP (CORRECT - +1.0))
- **Câu hỏi gốc**: trong thang 9/2025, cua hang co may don o trang thai huy
- **Ngữ cảnh**: `as_of_date = 2026-09-03` | `intent = cancelled_orders` | Khoảng ngày: `2025-09-01 -> 2025-09-30` (Tháng 9/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date', 'Mã đơn hàng']
- **Dự đoán (Pred)**: `23` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `23` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.90` | **Số vòng tự sửa lỗi (Self-Correction)**: `1`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`252.4ms` | Grounding=`46.8ms` | LLM=`62258.2ms` | Thực thi=`51366.3ms` | Normalizer=`0.6ms` | Retry=`1`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Câu hỏi đếm đơn bị hủy nhưng code không lọc điều kiện trạng thái hủy` -> Kết quả thử lại: `Thành công`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df = get_sales_table(data)
    date_col = get_date_col(df)
    status_col = next((col for col in df.columns if 'trạng thái' in col.lower()), None)
    
    if not status_col:
        return 0
    
    df['clean_date'] = clean_date(df[date_col])
    filtered_df = df[(df['clean_date'] >= pd.Timestamp('2025-09-01')) & (df['clean_date'] <= pd.Timestamp('2025-09-30 23:59:59'))]
    
    cancelled_orders = filtered_df[filtered_df[status_col].str.contains('hủy', case=False, na=False)]
    
    return int(cancelled_orders['Mã đơn hàng'].dropna().nunique())
```
</details>

### 🛡️ Câu 24: `q_040c9dd7f4de` (`biz_116`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Tháng 9/2025 top 5 hàng bán chạy nhất là gì?
- **Ngữ cảnh**: `as_of_date = 2026-09-03` | `intent = ranking_topk` | Khoảng ngày: `2025-09-01 -> 2025-09-30` (Tháng 9/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `['Xi măng Hà Tiên PCB40 bao 50kg', 'Gạch ống 8x8x18 Tuynel', 'Xi măng INSEE Power-S bao 50kg', 'Băng keo điện Nano 20 yard', 'Ống PVC Bình Minh D32 cây 4m']` (Kiểu: `list`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`0.0ms` | Grounding=`47.1ms` | LLM=`45287.9ms` | Thực thi=`2776.0ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 25: `q_0496b83ca7c4` (`biz_116`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Doanh thu ngày nào cao nhất trong tháng 7 năm 2026?
- **Ngữ cảnh**: `as_of_date = 2026-09-03` | `intent = ranking_topk` | Khoảng ngày: `2026-07-01 -> 2026-07-31` (Tháng 7/2026)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `2026-07-29` (Kiểu: `date`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`239.7ms` | Grounding=`47.4ms` | LLM=`36193.1ms` | Thực thi=`20898.1ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 26: `q_047586efd4bf` (`biz_117`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Giúp mình đếm số đợt giao hàng trong tháng 1/2026.
- **Ngữ cảnh**: `as_of_date = 2026-05-20` | `intent = count_orders` | Khoảng ngày: `2026-01-01 -> 2026-01-31` (Tháng 1/2026)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `872` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`275.0ms` | Grounding=`34.2ms` | LLM=`44594.0ms` | Thực thi=`18861.5ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ❌ Câu 27: `q_050132b11a75` (`biz_117`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Cho mình hỏi ngày 19/03/2026 cửa hàng bán Nước kiềm Ion Life 19L với đơn giá bao nhiêu một bình?
- **Ngữ cảnh**: `as_of_date = 2026-05-20` | `intent = price_lookup` | Khoảng ngày: `2026-03-19 -> 2026-03-19` (Ngày 19/03/2026 (2026-03-19))
- **Thực thể khớp (Entity Linking)**: `Nước kiềm Ion Life 19L`, `Nước khoáng Lavie 19L`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date']
- **Dự đoán (Pred)**: `377500` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `75500` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`243.0ms` | Grounding=`81.6ms` | LLM=`131176.4ms` | Thực thi=`15476.0ms` | Normalizer=`0.9ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `302000.0` (400.0%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    target_date = pd.Timestamp('2026-03-19')
    
    # BƯỚC 1: Tra cứu trong bảng bán hàng
    for name, df in data.items():
        if any(k in str(name).lower() for k in ['nhap', 'kho', 'chiphi', 'ncc']):
            continue
        d_c = get_date_col(df)
        a_c = get_amount_col(df)
        i_c = get_item_col(df)
        
        if d_c and a_c and i_c:
            m_prod = df[i_c].astype(str).str.contains('Nước kiềm Ion Life 19L', case=False, na=False)
            matched = df[m_prod].copy()
            if not matched.empty:
                matched['clean_date'] = clean_date(matched[d_c])
                matched_day = matched[(matched['clean_date'].dt.date == target_date)]
                if not matched_day.empty:
                    return int(round(clean_num(matched_day[a_c]).iloc[0]))
    
    # BƯỚC 2: Nếu không có giao dịch trong ngày, tra cứu Bảng giá
    for name, df in data.items():
        cols = df.columns.tolist()
        i_c = next((c for c in cols if any(k in str(c).lower() for k in ['tên hàng', 'hàng', 'tên món', 'sản phẩm', 'mặt hàng'])), None)
        p_c = next((c for c in cols if any(k in str(c).lower() for k in ['giá mới', 'giá bán', 'giá', 'đơn giá', 'giá lẻ'])), None)
        if i_c and p_c:
            m_prod = df[i_c].astype(str).str
```
</details>

### 🛡️ Câu 28: `q_028dcfb561ea` (`biz_118`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Tháng 6/2026 khách mua từ 4 đơn trở lên chiếm bao nhiêu % số khách đã mua nhỉ?
- **Ngữ cảnh**: `as_of_date = 2026-09-10` | `intent = rate_percentage` | Khoảng ngày: `2026-06-01 -> 2026-06-30` (Tháng 6/2026)
- **Thực thể khớp (Entity Linking)**: `Đá 4x6`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `23.77` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `2`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`281.7ms` | Grounding=`72.0ms` | LLM=`160139.8ms` | Thực thi=`153776.9ms` | Normalizer=`0.0ms` | Retry=`2`
- **⚠️ Nguyên nhân Lỗi / Lý do chặn**: `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 15, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
`
- **🛡️ Lý do Cổng Abstention**: `Runtime Error: Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 15, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
`
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Câu hỏi kiểu percent nhưng code không thực hiện phép chia tính tỷ lệ` -> Kết quả thử lại: `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 15, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
`
  - Vòng 2: Lỗi đầu vào `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 15, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
` -> Kết quả thử lại: `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 15, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Lấy bảng bán hàng chi tiết
    df = get_sales_table(data)
    
    # Lấy cột ngày, mã đơn hàng
    date_col = get_date_col(df)
    order_col = next((col for col in df.columns if 'Mã đơn hàng' in col), None)
    
    if not date_col or not order_col:
        return 0.0
    
    # Lọc dữ liệu trong tháng 6/2026
    start_date = pd.Timestamp('2026-06-01')
    end_date = pd.Timestamp('2026-06-30 23:59:59')
    filtered_df = df[(df[date_col] >= start_date) & (df[date_col] <= end_date)]
    
    # Đếm số đơn hàng của mỗi khách hàng
    customer_order_count = filtered_df.groupby('Khách hàng')[order_col].nunique()
    
    # Đếm số khách hàng mua từ 4 đơn trở lên
    customers_with_4_or_more_orders = (customer_order_count >= 4).sum()
    
    # Tổng số khách hàng
    total_customers = customer_order_count.shape[0]
    
    # Tính tỷ lệ phần trăm
    if total_customers == 0:
        return 0.0
    
    percentage = (customers_with_4_or_more_orders / total_customers) * 100
    
    return round(percentage, 2)
```
</details>

### 🛡️ Câu 29: `q_03766ac30505` (`biz_118`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Quý 4/2025 cửa hàng có bao nhiêu đơn hàng ạ?
- **Ngữ cảnh**: `as_of_date = 2026-09-10` | `intent = count_orders` | Khoảng ngày: `2025-10-01 -> 2025-12-31` (Quý 4/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `3238` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`250.2ms` | Grounding=`38.1ms` | LLM=`36190.1ms` | Thực thi=`24444.8ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ✅ Câu 30: `q_04128078842a` (`biz_118`) — Điểm PA: `+1.0` (HOÀN TOÀN TRÙNG KHỚP (CORRECT - +1.0))
- **Câu hỏi gốc**: cửa hàng có mấy khách nhà ở đường Hai Bà Trưng
- **Ngữ cảnh**: `as_of_date = 2026-09-10` | `intent = unanswerable_gate` | Khoảng ngày: `None -> None` ()
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `unanswerable`) | **Đáp án Chuẩn (Gold)**: `None` (Kiểu: `unanswerable`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`246.7ms` | Grounding=`0.0ms` | LLM=`0.0ms` | Thực thi=`0.0ms` | Normalizer=`0.0ms` | Retry=`0`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
# Feasibility Gate: Unanswerable detected deterministically
```
</details>

### ❌ Câu 31: `q_02b4a032b9f3` (`biz_120`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Trong tháng 11/2025, tỉ lệ đơn huỷ là bao nhiêu phần trăm?
- **Ngữ cảnh**: `as_of_date = 2026-06-12` | `intent = rate_percentage` | Khoảng ngày: `2025-11-01 -> 2025-11-30` (Tháng 11/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date']
- **Dự đoán (Pred)**: `1.28` (Kiểu: `percent`) | **Đáp án Chuẩn (Gold)**: `1.17` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.85` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`592.5ms` | Grounding=`81.2ms` | LLM=`22939.0ms` | Thực thi=`43547.6ms` | Normalizer=`1.0ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`50%` (Phạt `-0.15`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `0.11` (9.401709401709402%) — **Chẩn đoán**: *CÓ THỂ QUÊN LỌC ĐƠN HỦY / TRẢ LẠI (Giá trị dự đoán cao hơn chuẩn 5-30%).*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df_ord = None
    date_col = None
    status_col = None
    order_col = None
    for name, df in data.items():
        cols = df.columns.tolist()
        d_c = next((c for c in cols if any(k in str(c).lower() for k in ['ngày', 'date', 'ngaylap', 'thời gian'])), None)
        s_c = next((c for c in cols if any(k in str(c).lower() for k in ['trạng thái', 'status', 'tình trạng', 'ghi chú tt'])), None)
        if d_c and s_c:
            df_ord = df.copy()
            date_col = d_c
            status_col = s_c
            order_col = next((c for c in cols if any(k in str(c).lower() for k in ['mã đơn', 'số hđ', 'so hd', 'order id', 'mã giao dịch', 'mã phiếu', 'số phiếu'])), None)
            break

    if df_ord is None or not date_col or not status_col:
        return 0.0

    df_ord['clean_date'] = clean_date(df_ord[date_col])
    mask = (df_ord['clean_date'] >= pd.Timestamp("2026-06-01")) & (df_ord['clean_date'] <= pd.Timestamp("2026-06-30 23:59:59"))
    df_period = df_ord[mask]
    if df_period.empty:
        return 0.0

    cancel_mask = df_period[status_col].astype(str).str.lower().str.contains(r'hủy|huỷ|huy|cancel', na=False)
    if order_col:
        total_orders = df_period[order_col].dropna().nunique()
        cancelled_count = df_period[cancel_mask][order_col].dropna().nunique()
    else:
        total_orders = len(df_period)
        cancelled_count = int(cancel_mask.sum())

    if total_orders == 0:
        return 0.0
    rate = (cancelled_count / total_orders) * 100.0
    return round(float(rate), 2)
```
</details>

### ❌ Câu 32: `q_046b75d6192e` (`biz_120`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: xem giup ngay nao trong thang 2/2025 dthu cao nhat
- **Câu hỏi mở rộng viết tắt**: xem giup ngay nao trong thang 2/2025 doanh thu cao nhat
- **Ngữ cảnh**: `as_of_date = 2026-06-12` | `intent = ranking_topk` | Khoảng ngày: `2025-02-01 -> 2025-02-28` (Tháng 2/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date', 'clean_amt', 'day_str']
- **Dự đoán (Pred)**: `2025-08-23` (Kiểu: `date`) | **Đáp án Chuẩn (Gold)**: `2025-02-15` (Kiểu: `date`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`470.8ms` | Grounding=`97.2ms` | LLM=`22232.0ms` | Thực thi=`44687.3ms` | Normalizer=`0.6ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    dfs = [df for n, df in data.items() if any(k in str(n).lower() for k in ['banhang', 'donhang', 'hoadon', 'ban_hang', 'don_hang']) and 'chitiet' not in str(n).lower()]
    df_sales = pd.concat(dfs, ignore_index=True) if dfs else list(data.values())[0].copy()
    df_sales = filter_valid_sales(df_sales)

    cols = df_sales.columns.tolist()
    d_c = next((c for c in cols if any(k in str(c).lower() for k in ['ngày', 'date', 'thời gian'])), None)
    a_c = next((c for c in cols if any(k in str(c).lower() for k in ['tổng tiền hàng', 'thành tiền', 'trị giá', 'doanh thu', 'tổng tiền']) and 'thuế' not in str(c).lower()), None)

    df_sales['clean_date'] = clean_date(df_sales[d_c])
    df_sales['clean_amt'] = clean_num(df_sales[a_c])
    mask = (df_sales['clean_date'] >= pd.Timestamp("2025-08-01")) & (df_sales['clean_date'] <= pd.Timestamp("2025-08-31 23:59:59"))
    df_m = df_sales[mask].copy()
    df_m['day_str'] = df_m['clean_date'].dt.strftime('%Y-%m-%d')
    daily = df_m.groupby('day_str', as_index=False)['clean_amt'].sum()
    daily = daily.sort_values(by=['clean_amt', 'day_str'], ascending=[False, True])
    return str(daily.iloc[0]['day_str'])
```
</details>

### ✅ Câu 33: `q_03a446be8b75` (`biz_121`) — Điểm PA: `+1.0` (HOÀN TOÀN TRÙNG KHỚP (CORRECT - +1.0))
- **Câu hỏi gốc**: trong thang 2/2025, ti le don huy la bao nhieu %
- **Ngữ cảnh**: `as_of_date = 2026-09-20` | `intent = unanswerable_gate` | Khoảng ngày: `None -> None` ()
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `unanswerable`) | **Đáp án Chuẩn (Gold)**: `None` (Kiểu: `unanswerable`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`312.0ms` | Grounding=`0.0ms` | LLM=`0.0ms` | Thực thi=`0.0ms` | Normalizer=`0.0ms` | Retry=`0`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
# Feasibility Gate: Unanswerable detected deterministically
```
</details>

### ❌ Câu 34: `q_05b478fbe670` (`biz_122`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: cho hỏi so số viên magne-B6 corbière hộp 5 vỉ x 10 viên bán ra trong 7 ngày kể từ ngày 29/11/2025, ngày tăng giá lên 2.200đ, vs 7 ngày ngay trước ngày đó: tăng hay giảm bao nhiu %
- **Câu hỏi mở rộng viết tắt**: cho hỏi so số viên magne-B6 corbière hộp 5 vỉ x 10 viên bán ra trong 7 ngày kể từ ngày 29/11/2025, ngày tăng giá lên 2.200đ, với 7 ngày ngay trước ngày đó: tăng hay giảm bao nhiu %
- **Ngữ cảnh**: `as_of_date = 2026-07-17` | `intent = rate_percentage` | Khoảng ngày: `2025-11-29 -> 2025-12-05` (Kỳ chính: 7 ngày kể từ 2025-11-29 (đến 2025-12-05) | Kỳ đối chiếu: 7 ngày ngay trước 2025-11-29 (từ 2025-11-22 đến 2025-11-28)) | Kỳ so sánh: `2025-11-22 -> 2025-11-28` (7 ngày ngay trước 2025-11-29 (từ 2025-11-22 đến 2025-11-28))
- **Thực thể khớp (Entity Linking)**: `Magne-B6 Corbière hộp 5 vỉ x 10 viên`, `10`, `2,200`, `5,200`, `10,200`, `Enervon hộp 10 vỉ x 10 viên`, `Eugica hộp 10 vỉ x 10 viên nang`, `Gofen 400 hộp 5 vỉ x 10 viên nang`, `122,200`, `Alaxan hộp 25 vỉ x 4 viên`, `Loperamid 2mg Domesco hộp 5 vỉ x 10 viên`, `Loperamid  2mg Domesco hộp 5 vỉ x 10 viên`, `Clarityne 10mg hộp 1 vỉ x 10 viên`, `Boganic Traphaco hộp 5 vỉ x 20 viên`, `9,200`, `3,200`, `2,600`, `2,100`, `2,000`, `1,200`, `2,300`, `2,700`, `2,800`, `Zyrtec 10mg hộp 1 vỉ x 10 viên`, `Strepsils Cool hộp 2 vỉ x 12 viên`, `Enat 400 hộp 3 vỉ x 10 viên nang`, `ENAT 400 HỘP 3 VỈ X 10 VIÊN NANG`, `Augmentin 625mg hộp 2 vỉ x 7 viên`, `51,200`, `19,200`, `43,200`, `Amlodipin 5mg Domesco hộp 3 vỉ x 10 viên`, `Cetirizin 10mg Domesco hộp 10 vỉ x 10 viên`, `Panadol Extra hộp 15 vỉ x 12 viên`, `Decolgen Forte hộp 30 vỉ x 4 viên`, `Bisolvon 8mg hộp 3 vỉ x 10 viên`, `Tiffy Dey hộp 25 vỉ x 4 viên`, `Lorastad 10mg Stella hộp 10 vỉ x 10 viên`, `Air-X 80mg viên nhai hộp 10 vỉ x 10 viên`, `Lorastad 10mg Stella Hộp 10 Vỉ X 10 Viên`, `5,100`, `Hagifen 400mg DHG hộp 10 vỉ x 10 viên`, `Amoxicillin 500mg Domesco hộp 10 vỉ x 10 viên`, `Amoxicillin 500mg Domesco Hộp 10 Vỉ X 10 Viên`, `Panadol 500mg hộp 10 vỉ x 12 viên`, `Panadol 500mg Hộp 10 Vỉ X 12 Viên`, `PANADOL 500MG HỘP 10 VỈ X 12 VIÊN`, `Cefuroxim 500mg Vidipha hộp 10 vỉ x 10 viên`, `Cefuroxim 500mg Vidipha Hộp 10 Vỉ X 10 Viên`, `Metformin Stella 500mg hộp 6 vỉ x 10 viên`, `Metformin Stella 500mg hop 6 vi x 10 vien`, `5,000`, `5,300`, `7,500`, `7,000`, `7,900`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date', 'clean_qty']
- **Dự đoán (Pred)**: `140.74` (Kiểu: `percent`) | **Đáp án Chuẩn (Gold)**: `123.46` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.64` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`33246.1ms` | Grounding=`1045.9ms` | LLM=`48757.0ms` | Thực thi=`36760.2ms` | Normalizer=`1.3ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`25%` (Phạt `-0.26`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `17.28` (13.996436092661591%) — **Chẩn đoán**: *CÓ THỂ QUÊN LỌC ĐƠN HỦY / TRẢ LẠI (Giá trị dự đoán cao hơn chuẩn 5-30%).*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df_sales = get_sales_table(data)
    d_c = get_date_col(df_sales)
    q_c = get_qty_col(df_sales)
    p_c = get_item_col(df_sales)
    
    df_sales['clean_date'] = clean_date(df_sales[d_c])
    df_sales['clean_qty'] = clean_num(df_sales[q_c])
    df_prod = df_sales[df_sales[p_c].astype(str).str.contains('Magne-B6 Corbière hộp 5 vỉ x 10 viên', case=False, na=False)]
    
    m_after = (df_prod['clean_date'] >= pd.Timestamp('2025-11-29')) & (df_prod['clean_date'] <= pd.Timestamp('2025-12-05 23:59:59'))
    m_before = (df_prod['clean_date'] >= pd.Timestamp('2025-11-22')) & (df_prod['clean_date'] <= pd.Timestamp('2025-11-28 23:59:59'))
    
    qty_after = df_prod[m_after]['clean_qty'].sum()
    qty_before = df_prod[m_before]['clean_qty'].sum()
    if qty_before == 0:
        return 0.0
    return round(float(((qty_after - qty_before) / qty_before) * 100.0), 2)
```
</details>

### ❌ Câu 35: `q_026fa17395ed` (`biz_124`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Tính đến hết ngày 25/05/2026, khách hàng còn nợ cửa hàng tổng cộng bao nhiêu tiền?
- **Ngữ cảnh**: `as_of_date = 2026-05-25` | `intent = cong_no` | Khoảng ngày: `2026-05-25 -> 2026-05-25` (Ngày 25/05/2026 (2026-05-25))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `14298400` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `948973` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.70` | **Số vòng tự sửa lỗi (Self-Correction)**: `2`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`884.0ms` | Grounding=`123.7ms` | LLM=`107859.6ms` | Thực thi=`146694.2ms` | Normalizer=`0.8ms` | Retry=`2`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `13349427.0` (1406.7235843380158%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Câu hỏi hỏi về công nợ / tiền trả nợ nhưng code chỉ tính doanh thu bán hàng chung mà không xét bảng/cột thanh toán hoặc công nợ` -> Kết quả thử lại: `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 7, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
`
  - Vòng 2: Lỗi đầu vào `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 7, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
` -> Kết quả thử lại: `Thành công`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df = get_sales_table(data)
    date_col = get_date_col(df)
    amount_col = get_amount_col(df)
    
    if not df.empty and date_col in df.columns and amount_col in df.columns:
        df[date_col] = clean_date(df[date_col])
        filtered_df = df[(df[date_col].dt.date == pd.to_datetime('2026-05-25').date())]
        total_debt = filtered_df[amount_col].sum()
        return int(total_debt)
    else:
        return 0
```
</details>

### ❌ Câu 36: `q_02b5f208ee57` (`biz_124`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Từ ngày 16/03/2025, Fugacar 500mg hộp 1 viên tăng giá lên 23.000đ. So 7 ngày kể từ ngày 16/03/2025 với 7 ngày ngay trước đó, số viên bán ra thay đổi bao nhiêu phần trăm nhỉ?
- **Ngữ cảnh**: `as_of_date = 2026-05-25` | `intent = rate_percentage` | Khoảng ngày: `2025-03-16 -> 2025-03-22` (Kỳ chính: 7 ngày kể từ 2025-03-16 (đến 2025-03-22) | Kỳ đối chiếu: 7 ngày ngay trước 2025-03-16 (từ 2025-03-09 đến 2025-03-15)) | Kỳ so sánh: `2025-03-09 -> 2025-03-15` (7 ngày ngay trước 2025-03-16 (từ 2025-03-09 đến 2025-03-15))
- **Thực thể khớp (Entity Linking)**: `Fugacar 500mg hộp 1 viên`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date', 'clean_qty']
- **Dự đoán (Pred)**: `0.0` (Kiểu: `percent`) | **Đáp án Chuẩn (Gold)**: `69.57` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.85` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`736.6ms` | Grounding=`833.0ms` | LLM=`58632.7ms` | Thực thi=`68415.0ms` | Normalizer=`3.0ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `69.57` (100.0%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df_sales = get_sales_table(data)
    if df_sales is None or df_sales.empty:
        return 0.0

    date_col = get_date_col(df_sales)
    qty_col = get_qty_col(df_sales)
    item_col = get_item_col(df_sales)
    if not date_col or not qty_col or not item_col:
        return 0.0

    df_sales['clean_date'] = clean_date(df_sales[date_col])
    df_sales['clean_qty'] = clean_num(df_sales[qty_col])

    mask_curr = (df_sales['clean_date'] >= pd.Timestamp('2025-03-16')) & (df_sales['clean_date'] <= pd.Timestamp('2025-03-22 23:59:59'))
    mask_prev = (df_sales['clean_date'] >= pd.Timestamp('2025-03-09')) & (df_sales['clean_date'] <= pd.Timestamp('2025-03-15 23:59:59'))

    df_curr = df_sales[(df_sales[item_col] == 'Fugacar 500mg hộp 1 viên') & mask_curr]
    df_prev = df_sales[(df_sales[item_col] == 'Fugacar 500mg hộp 1 viên') & mask_prev]

    qty_curr = df_curr['clean_qty'].sum()
    qty_prev = df_prev['clean_qty'].sum()

    if qty_prev == 0:
        return 0.0

    pct_change = ((qty_curr - qty_prev) / qty_prev) * 100.0
    return round(float(pct_change), 2)
```
</details>

### 🛡️ Câu 37: `q_02eb42db6665` (`biz_125`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: tháng 5/2025 hàng nào bán chạy nhất
- **Ngữ cảnh**: `as_of_date = 2026-05-11` | `intent = ranking_topk` | Khoảng ngày: `2025-05-01 -> 2025-05-31` (Tháng 5/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `Bia Tiger lon` (Kiểu: `string`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`129.1ms` | Grounding=`4.8ms` | LLM=`45610.9ms` | Thực thi=`6378.1ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 38: `q_04ad09f4f1fc` (`biz_125`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Cho hỏi doanh thu ngày nào cao nhất trong tháng 3 năm 2025?
- **Ngữ cảnh**: `as_of_date = 2026-05-11` | `intent = ranking_topk` | Khoảng ngày: `2025-03-01 -> 2025-03-31` (Tháng 3/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `2025-03-01` (Kiểu: `date`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`124.3ms` | Grounding=`8.1ms` | LLM=`29587.2ms` | Thực thi=`6184.4ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 39: `q_0285dfd1bb28` (`biz_127`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Doanh thu trung bình trên một đơn tháng 3/2025 là bao nhiêu ạ?
- **Ngữ cảnh**: `as_of_date = 2026-07-28` | `intent = aggregation` | Khoảng ngày: `2025-03-01 -> 2025-03-31` (Tháng 3/2025)
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `522915` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`385.5ms` | Grounding=`10.6ms` | LLM=`39796.3ms` | Thực thi=`12726.1ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 40: `q_02b7cc8b174d` (`biz_127`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Xem giúp tháng 2/2026 nhóm hải sản chiếm bao nhiêu % doanh thu?
- **Ngữ cảnh**: `as_of_date = 2026-07-28` | `intent = rate_percentage` | Khoảng ngày: `2026-02-01 -> 2026-02-28` (Tháng 2/2026)
- **Thực thể khớp (Entity Linking)**: `hải sản`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `9.37` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `2`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`343.2ms` | Grounding=`15.6ms` | LLM=`148821.6ms` | Thực thi=`151767.2ms` | Normalizer=`0.0ms` | Retry=`2`
- **⚠️ Nguyên nhân Lỗi / Lý do chặn**: `Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: None

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 19, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 234, in _resilient_getitem
    return orig_getitem(self, key)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/frame.py", line 4113, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3819, in get_loc
    raise KeyError(key) from err
KeyError: None
`
- **🛡️ Lý do Cổng Abstention**: `Runtime Error: Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: None

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 19, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 234, in _resilient_getitem
    return orig_getitem(self, key)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/frame.py", line 4113, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3819, in get_loc
    raise KeyError(key) from err
KeyError: None
`
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Silent Zero: Kết quả phần trăm bằng 0 bất thường` -> Kết quả thử lại: `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 12, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
`
  - Vòng 2: Lỗi đầu vào `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 12, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
` -> Kết quả thử lại: `Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: None

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 19, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 234, in _resilient_getitem
    return orig_getitem(self, key)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/frame.py", line 4113, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/base.py", line 3819, in get_loc
    raise KeyError(key) from err
KeyError: None
`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Lấy bảng chi tiết đơn hàng
    df = get_sales_table(data)
    
    # Lấy cột ngày và cột thành tiền
    date_col = get_date_col(df)
    amount_col = get_amount_col(df)
    
    # Chuyển đổi cột ngày sang datetime
    df[date_col] = clean_date(df[date_col])
    
    # Lọc dữ liệu trong tháng 2/2026
    start_date = '2026-02-01'
    end_date = '2026-02-28'
    df_filtered = df[(df[date_col] >= pd.Timestamp(start_date)) & (df[date_col] <= pd.Timestamp(end_date))]
    
    # Lọc dữ liệu nhóm hải sản
    item_col = get_item_col(df)
    df_hai_san = df_filtered[df[item_col].str.contains('hải sản', case=False, na=False)]
    
    # Tính doanh thu tổng và doanh thu nhóm hải sản
    rev_total = df_filtered[amount_col].sum()
    rev_group = df_hai_san[amount_col].sum()
    
    # Trả về tỷ lệ phần trăm
    if rev_total == 0:
        return 0.0
    return round(float(rev_group / rev_total * 100.0), 2)
```
</details>

### 🛡️ Câu 41: `q_044a0d4c3ff0` (`biz_127`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: tieenf bán má heo nướng nước mắm tháng 9/2025 là nhiêu
- **Câu hỏi mở rộng viết tắt**: tiền bán má heo nướng nước mắm tháng 9/2025 là nhiêu
- **Ngữ cảnh**: `as_of_date = 2026-07-28` | `intent = aggregation` | Khoảng ngày: `2025-09-01 -> 2025-09-30` (Tháng 9/2025)
- **Thực thể khớp (Entity Linking)**: `nướng`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `7623000` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`338.9ms` | Grounding=`9.4ms` | LLM=`38600.2ms` | Thực thi=`12101.7ms` | Normalizer=`0.2ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`0%` (Phạt `-0.35`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ❌ Câu 42: `q_046b262bd6a7` (`biz_127`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: so cùng kỳ năm trước, doanh thu thangs 5/2026 tawng/giamr mấy %
- **Câu hỏi mở rộng viết tắt**: so cùng kỳ năm trước, doanh thu tháng 5/2026 tăng/giảm mấy %
- **Ngữ cảnh**: `as_of_date = 2026-07-28` | `intent = rate_percentage` | Khoảng ngày: `2026-05-01 -> 2026-05-31` (Kỳ chính: Tháng 5/2026 | Kỳ đối chiếu: Cùng kỳ năm trước (từ 2025-05-01 đến 2025-05-31)) | Kỳ so sánh: `2025-05-01 -> 2025-05-31` (Cùng kỳ năm trước (từ 2025-05-01 đến 2025-05-31))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `7.85` (Kiểu: `percent`) | **Đáp án Chuẩn (Gold)**: `10.35` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.70` | **Số vòng tự sửa lỗi (Self-Correction)**: `2`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`355.8ms` | Grounding=`11.4ms` | LLM=`156364.8ms` | Thực thi=`134453.0ms` | Normalizer=`0.7ms` | Retry=`2`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `2.5` (24.154589371980677%) — **Chẩn đoán**: *CÓ THỂ LỌC THIẾU DÒNG HOẶC ÁP DỤNG TRÙNG ĐIỀU KIỆN (Dự đoán thấp hơn chuẩn 5-30%).*
- **🔄 Nhật ký Tự sửa lỗi (Self-Correction)**:
  - Vòng 1: Lỗi đầu vào `Câu hỏi kiểu percent nhưng code không thực hiện phép chia tính tỷ lệ` -> Kết quả thử lại: `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 7, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
`
  - Vòng 2: Lỗi đầu vào `Traceback (most recent call last):
  File "/kaggle/working/datamate/src/datamate/llm_execution/worker.py", line 922, in execute_code_in_env
    _raw = namespace["solution"](smart_data)
  File "<submitted_code>", line 7, in solution
  File "/kaggle/working/datamate/src/datamate/llm_execution/sandbox.py", line 826, in _wrapped
    return orig_op(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py", line 60, in __ge__
    return self._cmp_method(other, operator.ge)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/series.py", line 6138, in _cmp_method
    res_values = ops.comparison_op(lvalues, rvalues, op)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 344, in comparison_op
    res_values = comp_method_OBJECT_ARRAY(op, lvalues, rvalues)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/ops/array_ops.py", line 129, in comp_method_OBJECT_ARRAY
    result = libops.scalar_compare(x.ravel(), y, op)
  File "pandas/_libs/ops.pyx", line 107, in pandas._libs.ops.scalar_compare
TypeError: '>=' not supported between instances of 'str' and 'Timestamp'
` -> Kết quả thử lại: `Thành công`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df = get_sales_table(data)
    d_c = get_date_col(df)
    amt_c = get_amount_col(df)
    
    if not df.empty and d_c in df.columns and amt_c in df.columns:
        df[d_c] = clean_date(df[d_c])
        
        current_period = (df[d_c].dt.date >= pd.to_datetime('2026-05-01').date()) & (df[d_c].dt.date <= pd.to_datetime('2026-05-31').date())
        previous_period = (df[d_c].dt.date >= pd.to_datetime('2025-05-01').date()) & (df[d_c].dt.date <= pd.to_datetime('2025-05-31').date())
        
        current_revenue = df[current_period][amt_c].sum()
        previous_revenue = df[previous_period][amt_c].sum()
        
        if previous_revenue == 0:
            return 0.0
        
        percentage_change = ((current_revenue - previous_revenue) / previous_revenue) * 100
        return round(percentage_change, 2)
    
    return 0.0
```
</details>

### 🛡️ Câu 43: `q_051bb0f610e4` (`biz_127`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: tháng này, dthu các ddown trả bằng cod và các đơn trả bằng tm mỗi loại bao nhiêu
- **Câu hỏi mở rộng viết tắt**: tháng này, doanh thu các đơn trả bằng cod và các đơn trả bằng tm mỗi loại bao nhiêu
- **Ngữ cảnh**: `as_of_date = 2026-07-28` | `intent = aggregation` | Khoảng ngày: `2026-07-01 -> 2026-07-28` (Tháng này (từ 2026-07-01 đến 2026-07-28))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `[52063500, 432338000]` (Kiểu: `list`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`338.8ms` | Grounding=`17.0ms` | LLM=`41691.7ms` | Thực thi=`12147.5ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ❌ Câu 44: `q_0283688e44df` (`biz_128`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Cho hỏi tuần trước riêng ca chiều bán được tổng cộng bao nhiêu tiền?
- **Ngữ cảnh**: `as_of_date = 2025-06-20` | `intent = aggregation` | Khoảng ngày: `2025-06-09 -> 2025-06-15` (Tuần trước (từ 2025-06-09 đến 2025-06-15))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `689096000` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `109181000` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`388.9ms` | Grounding=`46.2ms` | LLM=`15000.0ms` | Thực thi=`34115.6ms` | Normalizer=`0.4ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `579915000.0` (531.1501085353678%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    return revenue(data, start_date='2025-06-09', end_date='2025-06-15', as_of_date='2025-06-20')
```
</details>

### ❌ Câu 45: `q_04293d2ed907` (`biz_128`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: Ca chiều tuần trước doanh thu được bao nhiêu?
- **Ngữ cảnh**: `as_of_date = 2026-06-18` | `intent = aggregation` | Khoảng ngày: `2026-06-08 -> 2026-06-14` (Tuần trước (từ 2026-06-08 đến 2026-06-14))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `783620800` (Kiểu: `number`) | **Đáp án Chuẩn (Gold)**: `112133300` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`318.2ms` | Grounding=`41.7ms` | LLM=`13942.5ms` | Thực thi=`34303.7ms` | Normalizer=`0.3ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `671487500.0` (598.8296964416458%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    return revenue(data, start_date='2026-06-08', end_date='2026-06-14', as_of_date='2026-06-18')
```
</details>

### ❌ Câu 46: `q_049799837470` (`biz_128`) — Điểm PA: `-1.0` (SAI LỆCH ĐÁP ÁN (WRONG - -1.0))
- **Câu hỏi gốc**: trong tong doanh thu thang nay, phan cua khach nhom than thiet la may %
- **Ngữ cảnh**: `as_of_date = 2026-09-07` | `intent = rate_percentage` | Khoảng ngày: `2026-09-01 -> 2026-09-07` (Tháng này (từ 2026-09-01 đến 2026-09-07))
- **Thực thể khớp (Entity Linking)**: `thân thiết`
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=['clean_date']
- **Dự đoán (Pred)**: `2.2` (Kiểu: `percent`) | **Đáp án Chuẩn (Gold)**: `16.76` (Kiểu: `percent`)
- **Độ tin cậy ($C$)**: `0.65` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`330.3ms` | Grounding=`84.3ms` | LLM=`21838.6ms` | Thực thi=`31494.6ms` | Normalizer=`0.9ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`0%` (Phạt `-0.35`), Temporal Coverage=`100%` (Phạt `-0.00`)
- **🔍 Phân tích Độ lệch Số học**: Lệch tuyệt đối = `14.56` (86.87350835322195%) — **Chẩn đoán**: *Sai lệch logic lọc hoặc cột tính toán*

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    df_ord = None
    date_col = None
    status_col = None
    order_col = None
    for name, df in data.items():
        cols = df.columns.tolist()
        d_c = next((c for c in cols if any(k in str(c).lower() for k in ['ngày', 'date', 'ngaylap', 'thời gian'])), None)
        s_c = next((c for c in cols if any(k in str(c).lower() for k in ['trạng thái', 'status', 'tình trạng', 'ghi chú tt'])), None)
        if d_c and s_c:
            df_ord = df.copy()
            date_col = d_c
            status_col = s_c
            order_col = next((c for c in cols if any(k in str(c).lower() for k in ['mã đơn', 'số hđ', 'so hd', 'order id', 'mã giao dịch', 'mã phiếu', 'số phiếu'])), None)
            break

    if df_ord is None or not date_col or not status_col:
        return 0.0

    df_ord['clean_date'] = clean_date(df_ord[date_col])
    mask = (df_ord['clean_date'] >= pd.Timestamp("2026-06-01")) & (df_ord['clean_date'] <= pd.Timestamp("2026-06-30 23:59:59"))
    df_period = df_ord[mask]
    if df_period.empty:
        return 0.0

    cancel_mask = df_period[status_col].astype(str).str.lower().str.contains(r'hủy|huỷ|huy|cancel', na=False)
    if order_col:
        total_orders = df_period[order_col].dropna().nunique()
        cancelled_count = df_period[cancel_mask][order_col].dropna().nunique()
    else:
        total_orders = len(df_period)
        cancelled_count = int(cancel_mask.sum())

    if total_orders == 0:
        return 0.0
    rate = (cancelled_count / total_orders) * 100.0
    return round(float(rate), 2)
```
</details>

### 🛡️ Câu 47: `q_05cefa2459c7` (`biz_129`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: soos đơn hoom nay là bao nhiêu
- **Câu hỏi mở rộng viết tắt**: số đơn hôm nay là bao nhiêu
- **Ngữ cảnh**: `as_of_date = 2026-04-22` | `intent = aggregation` | Khoảng ngày: `2026-04-22 -> 2026-04-22` (Hôm nay (2026-04-22))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `61` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`9633.8ms` | Grounding=`22.4ms` | LLM=`36356.3ms` | Thực thi=`13024.3ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 48: `q_057830855dd5` (`biz_130`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: Trung bình mỗi đơn tuần trước mang về bao nhiêu dthu?
- **Câu hỏi mở rộng viết tắt**: Trung bình mỗi đơn tuần trước mang về bao nhiêu doanh thu?
- **Ngữ cảnh**: `as_of_date = 2026-02-03` | `intent = average_order_value` | Khoảng ngày: `2026-01-26 -> 2026-02-01` (Tuần trước (từ 2026-01-26 đến 2026-02-01))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `549956` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`43792.4ms` | Grounding=`24.0ms` | LLM=`37846.6ms` | Thực thi=`44929.5ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### 🛡️ Câu 49: `q_058bd677f168` (`biz_130`) — Điểm PA: `+0.0` (TỪ CHỐI TRẢ LỜI (ABSTAIN - 0.0))
- **Câu hỏi gốc**: ê trung bình mỗi đơn tuần trc mang về bao nhiu dthu
- **Câu hỏi mở rộng viết tắt**: ê trung bình mỗi đơn tuần trước mang về bao nhiu doanh thu
- **Ngữ cảnh**: `as_of_date = 2025-04-08` | `intent = average_order_value` | Khoảng ngày: `2025-03-31 -> 2025-04-06` (Tuần trước (từ 2025-03-31 đến 2025-04-06))
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `abstain`) | **Đáp án Chuẩn (Gold)**: `536658` (Kiểu: `number`)
- **Độ tin cậy ($C$)**: `0.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`0.0ms` | Grounding=`16.5ms` | LLM=`38588.1ms` | Thực thi=`4067.8ms` | Normalizer=`0.1ms` | Retry=`0`
- **Độ bám dữ liệu (Grounding)**: Entity Coverage=`100%` (Phạt `-0.00`), Temporal Coverage=`0%` (Phạt `-0.25`)

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
def solution(data):
    # Fallback an toàn khi mã sinh ra không đạt chuẩn cú pháp
    return None
```
</details>

### ✅ Câu 50: `q_05a8b30443a1` (`biz_130`) — Điểm PA: `+1.0` (HOÀN TOÀN TRÙNG KHỚP (CORRECT - +1.0))
- **Câu hỏi gốc**: trong so khach da mua hang thang 7/2026, bnhiu % mua tu 8 don tro len
- **Ngữ cảnh**: `as_of_date = 2026-08-29` | `intent = unanswerable_gate` | Khoảng ngày: `None -> None` ()
- **Thực thể khớp (Entity Linking)**: *(Không có)*
- **Bảng & Cột truy vấn**: Bảng=[] | Cột=[]
- **Dự đoán (Pred)**: `None` (Kiểu: `unanswerable`) | **Đáp án Chuẩn (Gold)**: `None` (Kiểu: `unanswerable`)
- **Độ tin cậy ($C$)**: `1.00` | **Số vòng tự sửa lỗi (Self-Correction)**: `0`
- **Thời gian từng pha (ms)**: Nạp dữ liệu=`295.5ms` | Grounding=`0.0ms` | LLM=`0.0ms` | Thực thi=`0.0ms` | Normalizer=`0.0ms` | Retry=`0`

<details><summary><b>Xem mã nguồn Python sinh ra (Code)</b></summary>

```python
# Feasibility Gate: Unanswerable detected deterministically
```
</details>


In [ ]:
# ==============================================================================
# BƯỚC 6: SUY LUẬN CHÍNH THỨC TRÊN PUBLIC TEST (XUẤT SUBMISSION.JSONL)
# ==============================================================================
# Chạy theo thứ tự tuần tự từ câu 1 đến câu 1000 (nếu PRESERVE_ORDER = True).
# Hỗ trợ cờ --resume: nếu dừng hoặc bị ngắt, chạy lại sẽ tự động làm tiếp từ câu chưa xong.
import os
import gc
import torch
from pathlib import Path

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

%cd {PROJECT_ROOT}
Path(SUBMISSION_PATH).parent.mkdir(parents=True, exist_ok=True)
pub_log_dir = Path(RUN_DIR) / "public_test_logs"
pub_log_dir.mkdir(parents=True, exist_ok=True)

if not (Path("competition_data/public_test/questions.jsonl").exists()):
    print("❌ LỖI: Chưa có competition_data/public_test/questions.jsonl! Hãy kiểm tra lại Bước 3.")
else:
    flags = ["--resume"]
    if PUBLIC_RANGE:
        flags.append(f"--range {PUBLIC_RANGE}")
    elif PUBLIC_BATCH_SIZE is not None and 0 < int(PUBLIC_BATCH_SIZE) < 1000:
        flags.append(f"--limit {PUBLIC_BATCH_SIZE}")
    if PRESERVE_ORDER:
        flags.append("--preserve_order")
    extra_flags = " ".join(flags)

    if PUBLIC_RANGE:
        target_count = f"Khoảng câu hỏi {PUBLIC_RANGE}"
    else:
        target_count = "Toàn bộ 1.000 câu" if (PUBLIC_BATCH_SIZE is None or int(PUBLIC_BATCH_SIZE) <= 0 or int(PUBLIC_BATCH_SIZE) >= 1000) else f"{PUBLIC_BATCH_SIZE} câu đầu tiên"
    print(f"🚀 Bắt đầu suy luận nộp bài trên public_test: {target_count}...")
    print(f"📌 Mô hình: {MODEL_ID} ({QUANTIZATION}) | Thứ tự: {'TUẦN TỰ từ câu 1 -> 1000' if PRESERVE_ORDER else 'GOM NHÓM theo shop'}")
    
    predict_cmd = (
        f"python predict.py "
        f"--data_dir competition_data/public_test "
        f"--model_id '{MODEL_ID}' "
        f"--quantization '{QUANTIZATION}' "
        f"--output_file '{SUBMISSION_PATH}' "
        f"--log_dir '{pub_log_dir}' "
        f"--tau_star {TAU_STAR} "
        f"--timeout {TIMEOUT_S} "
        f"--seed 42 "
        f"{extra_flags}"
    )
    !{predict_cmd}

    print(f"✅ Quá trình suy luận hoàn tất! Kết quả đã ghi tại: {SUBMISSION_PATH}")

In [ ]:
# ==============================================================================
# BƯỚC 7: THẨM ĐỊNH TỰ ĐỘNG & HƯỚNG DẪN TẢI FILE SUBMISSION.JSONL NỘP BÀI
# ==============================================================================
import json
from collections import Counter
from pathlib import Path
from datamate.output_abstention.submission import validate_submission

%cd {PROJECT_ROOT}
submission_file = Path(SUBMISSION_PATH)
questions_file = Path(PROJECT_ROOT) / "competition_data/public_test/questions.jsonl"

if not submission_file.exists():
    print(f"❌ Chưa tìm thấy tệp {submission_file}. Hãy kiểm tra lại Bước 6!")
elif not questions_file.exists():
    print(f"❌ Chưa tìm thấy tệp {questions_file} để đối chiếu.")
else:
    all_qids = [json.loads(line)["id"] for line in open(questions_file, encoding="utf-8") if line.strip()]
    done_records = {}
    for line in open(submission_file, encoding="utf-8"):
        if line.strip():
            rec = json.loads(line)
            done_records[rec["id"]] = rec

    print(f"📌 Tiến độ hồ sơ nộp bài: {len(done_records)} / {len(all_qids)} câu hỏi ({100*len(done_records)/len(all_qids):.1f}%)")

    errors = validate_submission(submission_file, questions_file)
    format_errors = [e for e in errors if not e.startswith("Thiếu ")]
    missing_errors = [e for e in errors if e.startswith("Thiếu ")]

    if not format_errors:
        print("🎉 CẤU TRÚC ĐÁP ÁN: 100% câu hỏi đã hoàn thành đều đạt chuẩn cấu trúc của Ban Tổ Chức!")
    else:
        print(f"⚠️ Phát hiện {len(format_errors)} vấn đề định dạng:")
        for err in format_errors[:10]:
            print(f"  • {err}")

    if not missing_errors:
        print("🎉 TOÀN VẸN: Đã hoàn thành đủ 1.000 / 1.000 câu hỏi, sẵn sàng nộp bài chấm điểm chính thức!")
    else:
        print(f"ℹ️ Trạng thái số lượng câu: {missing_errors[0]}")

    type_counts = Counter(r.get("type", "unknown") for r in done_records.values())
    print("\n📊 Phân bố các dạng đáp án trong submission.jsonl:")
    for t_name, count in type_counts.most_common():
        print(f"  • {t_name:<16}: {count:>5} câu ({100*count/max(1, len(done_records)):5.1f}%)")

    print("\n📥 HƯỚNG DẪN TẢI FILE NỘP BÀI TỪ KAGGLE:")
    print(f"1. Nhìn sang cột bên phải (Notebook Sidebar) -> Mục 'Output'.")
    print(f"2. Tìm tệp '{submission_file.name}' trong thư mục /kaggle/working/.")
    print("3. Bấm vào biểu tượng 3 chấm (...) bên cạnh tệp và chọn 'Download'.")